# POS Agent Revenue Insights

## Notebook 2: Synthetic Data Generation

This notebook generates a realistic synthetic dataset for a Nigerian POS ecosystem.

The notebook creates:

- Calendar Dimension
- Provider Dimension
- State Dimension
- LGA Dimension
- Agent Dimension
- Transaction Type Dimension
- Daily Operating Cost Fact
- Transaction Fact (1,000,000+ records)

The final outputs will be exported as CSV files for Power BI.

In [8]:

# Imports


import os
import random
from datetime import datetime

import numpy as np
import pandas as pd

from faker import Faker
from tqdm.auto import tqdm

# Reproducibility
random.seed(42)
np.random.seed(42)

fake = Faker("en_NG")
Faker.seed(42)

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [9]:


# Project Configuration


START_DATE = "2024-01-01"
END_DATE = "2024-12-31"

NUM_TRANSACTIONS = 1_000_000

PROJECT_FOLDER = "."

RAW_FOLDER = os.path.join(PROJECT_FOLDER, "data", "raw")
PROCESSED_FOLDER = os.path.join(PROJECT_FOLDER, "data", "processed")

os.makedirs(RAW_FOLDER, exist_ok=True)
os.makedirs(PROCESSED_FOLDER, exist_ok=True)

print("Project folders ready.")

Project folders ready.


In [10]:

# Calendar Dimension

calendar = pd.DataFrame({

    "Date": pd.date_range(
        START_DATE,
        END_DATE,
        freq="D"
    )

})

calendar["Year"] = calendar["Date"].dt.year
calendar["Quarter"] = calendar["Date"].dt.quarter
calendar["Month"] = calendar["Date"].dt.month
calendar["Month_Name"] = calendar["Date"].dt.month_name()

calendar["Week"] = calendar["Date"].dt.isocalendar().week

calendar["Day"] = calendar["Date"].dt.day

calendar["Day_Name"] = calendar["Date"].dt.day_name()

calendar["Weekend"] = calendar["Day_Name"].isin(
    ["Saturday", "Sunday"]
)

calendar.head()

,Date,Year,Quarter,Month,Month_Name,Week,Day,Day_Name,Weekend
0,2024-01-01,2024,1,1,January,1,1,Monday,False
1,2024-01-02,2024,1,1,January,1,2,Tuesday,False
2,2024-01-03,2024,1,1,January,1,3,Wednesday,False
3,2024-01-04,2024,1,1,January,1,4,Thursday,False
4,2024-01-05,2024,1,1,January,1,5,Friday,False


In [11]:
calendar.to_csv(

    os.path.join(RAW_FOLDER, "dim_date.csv"),

    index=False

)

print(f"Calendar created ({len(calendar)} rows)")

Calendar created (366 rows)


In [12]:

# POS Providers


provider_df = pd.DataFrame({

    "Provider":[

        "Moniepoint",
        "OPay",
        "PalmPay",
        "FirstMonie",
        "Paga"

    ],

    "Market_Share":[

        0.34,
        0.29,
        0.18,
        0.11,
        0.08

    ]

})

provider_df

,Provider,Market_Share
0,Moniepoint,0.34
1,OPay,0.29
2,PalmPay,0.18
3,FirstMonie,0.11
4,Paga,0.08


In [13]:
provider_df.to_csv(

    os.path.join(RAW_FOLDER, "dim_provider.csv"),

    index=False

)

provider_market_share = dict(

    zip(

        provider_df.Provider,

        provider_df.Market_Share

    )

)

print("Provider dimension saved.")

Provider dimension saved.


In [14]:

# Nigerian States


state_data = [

["Lagos","South West",23000000,0.95,100,2500],
["FCT","North Central",3800000,0.92,95,900],
["Rivers","South South",8200000,0.82,90,700],
["Kano","North West",17000000,0.78,88,800],
["Oyo","South West",8500000,0.74,82,600],
["Ogun","South West",7000000,0.72,80,500],
["Kaduna","North West",9200000,0.69,77,480],
["Delta","South South",6200000,0.71,76,420],
["Anambra","South East",6200000,0.76,83,400],
["Akwa Ibom","South South",5600000,0.68,74,380],
["Abia","South East",4200000,0.64,70,260],
["Adamawa","North East",4800000,0.52,61,220],
["Bauchi","North East",7800000,0.49,58,250],
["Bayelsa","South South",2500000,0.67,73,180],
["Benue","North Central",6100000,0.54,64,230],
["Borno","North East",6200000,0.43,50,180],
["Cross River","South South",4200000,0.59,67,210],
["Ebonyi","South East",3200000,0.57,65,180],
["Edo","South South",5100000,0.66,72,260],
["Ekiti","South West",3200000,0.61,68,180],
["Enugu","South East",4500000,0.71,78,260],
["Gombe","North East",3600000,0.48,57,170],
["Imo","South East",6200000,0.73,79,280],
["Jigawa","North West",7000000,0.46,55,180],
["Kebbi","North West",5200000,0.44,53,160],
["Kogi","North Central",4500000,0.55,63,190],
["Kwara","North Central",3600000,0.63,69,200],
["Nasarawa","North Central",3000000,0.60,66,170],
["Niger","North Central",6800000,0.50,60,220],
["Ondo","South West",5200000,0.65,71,260],
["Osun","South West",4800000,0.64,70,240],
["Plateau","North Central",4500000,0.58,66,210],
["Sokoto","North West",5900000,0.45,54,170],
["Taraba","North East",3600000,0.47,56,160],
["Yobe","North East",3900000,0.42,49,150],
["Zamfara","North West",5200000,0.43,51,160]
]

location_df = pd.DataFrame(

    state_data,

    columns=[

        "State",
        "Geo_Zone",
        "Population",
        "Urbanization_Index",
        "Business_Activity_Index",
        "Estimated_Agents"

    ]

)

location_df.head()

,State,Geo_Zone,Population,Urbanization_Index,Business_Activity_Index,Estimated_Agents
0,Lagos,South West,23000000,0.95,100,2500
1,FCT,North Central,3800000,0.92,95,900
2,Rivers,South South,8200000,0.82,90,700
3,Kano,North West,17000000,0.78,88,800
4,Oyo,South West,8500000,0.74,82,600


# LGA Dimension

This section creates the location hierarchy used throughout the project.

Hierarchy:

Nigeria

→ State

→ Local Government Area (LGA)

Each LGA belongs to exactly one State.

Each Agent belongs to one LGA.

This creates a proper geographical hierarchy for Power BI drill-down analysis.

In [16]:

# State Coordinates


state_coordinates = {

"Lagos":(6.5244,3.3792),
"FCT":(9.0765,7.3986),
"Rivers":(4.8156,7.0498),
"Kano":(12.0022,8.5920),
"Oyo":(7.3775,3.9470),
"Ogun":(7.1608,3.3486),
"Kaduna":(10.5105,7.4165),
"Delta":(5.7040,5.9339),
"Anambra":(6.2209,6.9369),
"Akwa Ibom":(5.0077,7.8493),

"Abia":(5.4527,7.5248),
"Adamawa":(9.3265,12.3984),
"Bauchi":(10.3158,9.8442),
"Bayelsa":(4.7719,6.0699),
"Benue":(7.3369,8.7404),
"Borno":(11.8846,13.1510),
"Cross River":(5.9631,8.3345),
"Ebonyi":(6.2649,8.0137),
"Edo":(6.6342,5.9304),
"Ekiti":(7.7189,5.3110),
"Enugu":(6.4584,7.5464),
"Gombe":(10.2897,11.1673),
"Imo":(5.5720,7.0588),
"Jigawa":(12.2280,9.5616),
"Kebbi":(12.4539,4.1975),
"Kogi":(7.7337,6.6906),
"Kwara":(8.9669,4.3874),
"Nasarawa":(8.5380,8.3220),
"Niger":(9.9309,5.5983),
"Ondo":(7.2508,5.2103),
"Osun":(7.5629,4.5200),
"Plateau":(9.2182,9.5179),
"Sokoto":(13.0609,5.2390),
"Taraba":(7.9994,10.7730),
"Yobe":(12.2939,11.4390),
"Zamfara":(12.1704,6.6641)

}

print("Coordinates Loaded")

Coordinates Loaded


In [17]:

# Business Categories


business_categories={

"Standalone POS Kiosk":0.34,

"Provision Store":0.18,

"Supermarket":0.11,

"Pharmacy":0.08,

"Fuel Station":0.06,

"Electronics Shop":0.05,

"Restaurant":0.05,

"Market Stall":0.07,

"Mini Mart":0.03,

"Mobile Agent":0.03

}

category_names=list(business_categories.keys())

category_probs=list(business_categories.values())

In [18]:

# Agent Tiers


agent_tiers={

"Bronze":0.45,

"Silver":0.30,

"Gold":0.18,

"Platinum":0.07

}

tier_names=list(agent_tiers.keys())

tier_probs=list(agent_tiers.values())

In [19]:

# KYC Levels

kyc_levels={

"Tier 1":0.20,

"Tier 2":0.50,

"Tier 3":0.30

}

kyc_names=list(kyc_levels.keys())

kyc_probs=list(kyc_levels.values())

In [20]:

# POS Terminal Types

terminal_types={

"Android POS":0.55,

"Traditional POS":0.28,

"mPOS":0.17

}

terminal_names=list(terminal_types.keys())

terminal_probs=list(terminal_types.values())

In [21]:

# Registration Dates


registration_dates=pd.date_range(

"2018-01-01",

"2024-12-31",

freq="D"

)

In [22]:

# Build LGA Dimension

lga_rows=[]

for _, row in location_df.iterrows():

    state=row["State"]

    num_lgas=max(5, min(20, int(row["Estimated_Agents"]/50)))

    base_names=[
        "Central",
        "North",
        "South",
        "East",
        "West",
        "Metro",
        "Urban",
        "Rural",
        "Market",
        "Industrial"
    ]

    lga_names=[]

    for name in base_names:
        lga_names.append(f"{state} {name}")

    while len(lga_names)<num_lgas:
        lga_names.append(f"{state} Zone {len(lga_names)+1}")

    for lga in lga_names[:num_lgas]:

        lga_rows.append({

            "State":state,

            "Geo_Zone":row["Geo_Zone"],

            "LGA":lga,

            "Business_Activity_Index":row["Business_Activity_Index"]

        })

lga_df=pd.DataFrame(lga_rows)

print(f"LGAs Created: {len(lga_df):,}")

lga_df.head()

LGAs Created: 252


,State,Geo_Zone,LGA,Business_Activity_Index
0,Lagos,South West,Lagos Central,100
1,Lagos,South West,Lagos North,100
2,Lagos,South West,Lagos South,100
3,Lagos,South West,Lagos East,100
4,Lagos,South West,Lagos West,100


In [23]:
lga_df.to_csv(

os.path.join(RAW_FOLDER,"dim_location.csv"),

index=False

)

print("LGA Dimension Saved")

LGA Dimension Saved


# Agent Dimension

This section generates the POS agent master table.

Each agent is assigned:

- Unique Agent ID
- Business Name
- Provider
- State
- LGA
- Geo Zone
- Business Category
- Agent Tier
- KYC Level
- Terminal Type
- Registration Date
- Years in Business
- Risk Score
- Average Daily Customers
- Latitude
- Longitude

This dimension will be used as the parent table for all POS transactions.

In [25]:

# Determine Total Number of Agents


TOTAL_AGENTS = int(location_df["Estimated_Agents"].sum())

print(f"Total Agents to Generate: {TOTAL_AGENTS:,}")

Total Agents to Generate: 13,010


In [26]:

# Generate Agent Master Table


agents = []

agent_counter = 1

for _, state_row in location_df.iterrows():

    state = state_row["State"]
    zone = state_row["Geo_Zone"]
    num_agents = int(state_row["Estimated_Agents"])

    state_lgas = lga_df[
        lga_df["State"] == state
    ]["LGA"].tolist()

    lat_center, lon_center = state_coordinates[state]

    providers = np.random.choice(
        provider_df["Provider"],
        size=num_agents,
        p=provider_df["Market_Share"]
    )

    for provider in providers:

        lga = random.choice(state_lgas)

        tier = np.random.choice(
            tier_names,
            p=tier_probs
        )

        category = np.random.choice(
            category_names,
            p=category_probs
        )

        kyc = np.random.choice(
            kyc_names,
            p=kyc_probs
        )

        terminal = np.random.choice(
            terminal_names,
            p=terminal_probs
        )

        registration_date = random.choice(registration_dates)

        years = (
            pd.Timestamp("2025-01-01") -
            registration_date
        ).days / 365

        if tier == "Bronze":
            avg_customers = max(8, int(np.random.normal(28, 6)))
        elif tier == "Silver":
            avg_customers = max(20, int(np.random.normal(55, 10)))
        elif tier == "Gold":
            avg_customers = max(40, int(np.random.normal(95, 15)))
        else:
            avg_customers = max(70, int(np.random.normal(170, 25)))

        risk_score = round(
            np.random.uniform(5, 95),
            1
        )

        latitude = lat_center + np.random.normal(0, 0.05)
        longitude = lon_center + np.random.normal(0, 0.05)

        agents.append({

            "Agent_ID":
            f"AG{agent_counter:06d}",

            "Business_Name":
            fake.company(),

            "Provider":
            provider,

            "State":
            state,

            "Geo_Zone":
            zone,

            "LGA":
            lga,

            "Business_Category":
            category,

            "Agent_Tier":
            tier,

            "KYC_Level":
            kyc,

            "Terminal_Type":
            terminal,

            "Registration_Date":
            registration_date,

            "Years_in_Business":
            round(years,1),

            "Risk_Score":
            risk_score,

            "Average_Daily_Customers":
            avg_customers,

            "Latitude":
            round(latitude,6),

            "Longitude":
            round(longitude,6)

        })

        agent_counter += 1

In [27]:
agents_df = pd.DataFrame(agents)

print(agents_df.shape)

agents_df.head()

(13010, 16)


,Agent_ID,Business_Name,Provider,State,Geo_Zone,LGA,Business_Category,Agent_Tier,KYC_Level,Terminal_Type,Registration_Date,Years_in_Business,Risk_Score,Average_Daily_Customers,Latitude,Longitude
0,AG000001,"Chukwu, Okonkwo and Balogun",OPay,Lagos,South West,Lagos East,Fuel Station,Gold,Tier 2,mPOS,2018-04-13,6.7,81.9,92,6.516836,3.437490
1,AG000002,Okafor and Sons,Paga,Lagos,South West,Lagos Market,Standalone POS Kiosk,Gold,Tier 3,Android POS,2020-09-30,4.3,50.9,91,6.557195,3.388937
2,AG000003,"Chukwu, Akinwale and Olawale",PalmPay,Lagos,South West,Lagos Rural,Provision Store,Bronze,Tier 2,Traditional POS,2019-07-26,5.4,31.9,23,6.544753,3.436469
3,AG000004,"Ojo, Adeyemi and Okonkwo",OPay,Lagos,South West,Lagos East,Standalone POS Kiosk,Bronze,Tier 3,Android POS,2024-02-12,0.9,94.8,24,6.572056,3.404854
4,AG000005,Abiola and Sons,Moniepoint,Lagos,South West,Lagos South,Standalone POS Kiosk,Bronze,Tier 3,mPOS,2024-08-15,0.4,72.8,24,6.545996,3.414055


In [28]:
#Validate the Agent Table

print()

print("Missing Values")
print("----------------")
print(agents_df.isna().sum())

print()

print("Duplicate Agent IDs")
print("-------------------")
print(agents_df["Agent_ID"].duplicated().sum())

print()

print("Providers")
print("----------")
print(agents_df["Provider"].value_counts())

print()

print("Agent Tiers")
print("-----------")
print(agents_df["Agent_Tier"].value_counts())

print()

print("Business Categories")
print("-------------------")
print(agents_df["Business_Category"].value_counts())


Missing Values
----------------
Agent_ID                   0
Business_Name              0
Provider                   0
State                      0
Geo_Zone                   0
LGA                        0
Business_Category          0
Agent_Tier                 0
KYC_Level                  0
Terminal_Type              0
Registration_Date          0
Years_in_Business          0
Risk_Score                 0
Average_Daily_Customers    0
Latitude                   0
Longitude                  0
dtype: int64

Duplicate Agent IDs
-------------------
0

Providers
----------
Provider
Moniepoint    4407
OPay          3843
PalmPay       2336
FirstMonie    1386
Paga          1038
Name: count, dtype: int64

Agent Tiers
-----------
Agent_Tier
Bronze      5870
Silver      3864
Gold        2402
Platinum     874
Name: count, dtype: int64

Business Categories
-------------------
Business_Category
Standalone POS Kiosk    4377
Provision Store         2337
Supermarket             1470
Pharmacy           

In [29]:
agents_df.to_csv(

    os.path.join(
        RAW_FOLDER,
        "dim_agent.csv"
    ),

    index=False

)

print("Agent Dimension Saved Successfully")

Agent Dimension Saved Successfully


In [30]:
#Quick Business Summary

print("="*50)

print(f"Total Agents : {len(agents_df):,}")

print(f"States       : {agents_df['State'].nunique()}")

print(f"LGAs         : {agents_df['LGA'].nunique()}")

print(f"Providers    : {agents_df['Provider'].nunique()}")

print(f"Average Daily Customers : {agents_df['Average_Daily_Customers'].mean():.1f}")

print("="*50)

Total Agents : 13,010
States       : 36
LGAs         : 252
Providers    : 5
Average Daily Customers : 57.4


# Transaction Engine Configuration

This section defines the business rules for generating
1,000,000 realistic POS transactions.

The engine simulates:

- Daily demand
- Peak business hours
- Salary periods
- Provider behaviour
- Transaction success
- Fraud
- Revenue
- Profitability

In [32]:
#Transaction Types

transaction_type_df = pd.DataFrame({

    "Transaction_Type":[
        "Cash Withdrawal",
        "Cash Deposit",
        "Bank Transfer",
        "Airtime",
        "Bill Payment"
    ],

    "Probability":[
        0.56,
        0.17,
        0.16,
        0.07,
        0.04
    ]

})

transaction_type_df

,Transaction_Type,Probability
0,Cash Withdrawal,0.56
1,Cash Deposit,0.17
2,Bank Transfer,0.16
3,Airtime,0.07
4,Bill Payment,0.04


In [33]:
transaction_type_df.to_csv(

    os.path.join(
        RAW_FOLDER,
        "dim_transaction_type.csv"
    ),

    index=False

)

print("Transaction Type Dimension Saved")

Transaction Type Dimension Saved


In [34]:
#Commission Rates

commission_rates = {

"Moniepoint":{

"Cash Withdrawal":0.0080,
"Cash Deposit":0.0055,
"Bank Transfer":0.0040,
"Airtime":0.0200,
"Bill Payment":0.0180

},

"OPay":{

"Cash Withdrawal":0.0076,
"Cash Deposit":0.0050,
"Bank Transfer":0.0038,
"Airtime":0.0190,
"Bill Payment":0.0170

},

"PalmPay":{

"Cash Withdrawal":0.0078,
"Cash Deposit":0.0052,
"Bank Transfer":0.0039,
"Airtime":0.0195,
"Bill Payment":0.0175

},

"FirstMonie":{

"Cash Withdrawal":0.0081,
"Cash Deposit":0.0058,
"Bank Transfer":0.0043,
"Airtime":0.0200,
"Bill Payment":0.0185

},

"Paga":{

"Cash Withdrawal":0.0084,
"Cash Deposit":0.0060,
"Bank Transfer":0.0045,
"Airtime":0.0210,
"Bill Payment":0.0190

}

}

In [35]:
#Processing_cost

processing_cost = {

"Cash Withdrawal":35,

"Cash Deposit":25,

"Bank Transfer":18,

"Airtime":10,

"Bill Payment":15

}

In [36]:
#Provider Failure Rates

failure_rates = {

"Moniepoint":0.012,

"OPay":0.017,

"PalmPay":0.016,

"FirstMonie":0.020,

"Paga":0.023

}

In [37]:
#Fraud_rates

fraud_rates = {

"Cash Withdrawal":0.004,

"Cash Deposit":0.002,

"Bank Transfer":0.003,

"Airtime":0.001,

"Bill Payment":0.001

}

In [38]:
#customer_segments

customer_segments = {

"Retail":0.72,

"SME":0.19,

"Corporate":0.06,

"VIP":0.03

}

segment_names=list(customer_segments.keys())

segment_probs=list(customer_segments.values())

In [39]:
#payment_methods

payment_methods = {

"Card":0.56,

"Transfer":0.21,

"USSD":0.14,

"Wallet":0.09

}

payment_names=list(payment_methods.keys())

payment_probs=list(payment_methods.values())

In [40]:
#Peak Hour Distribution

hour_distribution = {

8:0.04,

9:0.06,

10:0.08,

11:0.09,

12:0.09,

13:0.08,

14:0.08,

15:0.10,

16:0.12,

17:0.13,

18:0.08,

19:0.04,

20:0.01

}

hours=list(hour_distribution.keys())

hour_probs=list(hour_distribution.values())

In [41]:
#Prepare Weighted Agent Selection

agent_lookup = agents_df.copy()

agent_lookup["Selection_Weight"] = (

    agent_lookup["Average_Daily_Customers"]

    /

    agent_lookup["Average_Daily_Customers"].sum()

)


# Generate POS Transactions

This section creates the central fact table.

Each transaction represents one customer interaction at a POS agent.

The simulation incl- Profitability

In [43]:

# Generate Transactions (Chunk Engine)


CHUNK_SIZE = 100_000

NUM_CHUNKS = NUM_TRANSACTIONS // CHUNK_SIZE

print(f"Generating {NUM_TRANSACTIONS:,} transactions...")
print(f"Chunks: {NUM_CHUNKS}")

Generating 1,000,000 transactions...
Chunks: 10


In [44]:

# Agent Selection Weights


agent_lookup = agents_df.copy()

agent_lookup["Selection_Weight"] = (
    agent_lookup["Average_Daily_Customers"] /
    agent_lookup["Average_Daily_Customers"].sum()
)

agent_weights = agent_lookup["Selection_Weight"].values

agent_indices = np.arange(len(agent_lookup))

print("Agent weights ready.")

Agent weights ready.


In [45]:

# Calendar Dates


calendar_dates = pd.date_range(
    START_DATE,
    END_DATE,
    freq="D"
)

print(f"Calendar contains {len(calendar_dates)} days.")

Calendar contains 366 days.


In [46]:

# Hour Distribution


hour_distribution = {

    8:0.04,
    9:0.06,
    10:0.08,
    11:0.09,
    12:0.09,
    13:0.08,
    14:0.08,
    15:0.10,
    16:0.12,
    17:0.13,
    18:0.08,
    19:0.04,
    20:0.01

}

hours = list(hour_distribution.keys())

hour_probs = list(hour_distribution.values())

print("Hour distribution ready.")

Hour distribution ready.


In [47]:

#Validation
required = [
    "agent_lookup",
    "agent_weights",
    "agent_indices",
    "calendar_dates",
    "hours",
    "hour_probs",
    "commission_rates",
    "processing_cost",
    "failure_rates",
    "fraud_rates",
    "payment_names",
    "payment_probs",
    "segment_names",
    "segment_probs"
]

missing = []

for var in required:
    if var not in globals():
        missing.append(var)

if len(missing) == 0:
    print("All required variables are loaded.")
else:
    print("Missing variables:")
    print(missing)

All required variables are loaded.


In [48]:
import numpy as np
import pandas as pd


# Business Day Multiplier


def day_multiplier(date):

    if not isinstance(date, pd.Timestamp):
        date = pd.Timestamp(date)

    weekday = date.weekday()

    if weekday == 0:      # Monday
        return 1.18

    elif weekday == 4:    # Friday
        return 1.12

    elif weekday >= 5:    # Saturday/Sunday
        return 0.74

    return 1.00


# Salary Period


def salary_multiplier(date):

    if not isinstance(date, pd.Timestamp):
        date = pd.Timestamp(date)

    if date.day >= 25:
        return 1.25

    return 1.00



# Generate Transaction Amount


def generate_amount(transaction_type):

    ranges = {

        "Cash Withdrawal": (500, 5000, 100000),

        "Cash Deposit": (1000, 7000, 150000),

        "Bank Transfer": (500, 3000, 250000),

        "Airtime": (100, 1000, 10000),

        "Bill Payment": (200, 2500, 50000)

    }

    low, mode, high = ranges[transaction_type]

    return round(
        np.random.triangular(low, mode, high),
        2
    )



# Commission


def calculate_commission(provider, tx_type, amount):

    rate = commission_rates[provider][tx_type]

    return round(amount * rate, 2)



# Processing Cost


def calculate_processing_cost(tx_type):

    return processing_cost[tx_type]



# Transaction Status


def transaction_status(provider):

    if np.random.rand() < failure_rates[provider]:

        return "Failed"

    return "Successful"



# Fraud Status


def fraud_status(tx_type):

    if np.random.rand() < fraud_rates[tx_type]:

        return "Fraud"

    return "Clean"



# Agent Selection


agent_lookup = agents_df.copy()

agent_lookup["Selection_Weight"] = (
    agent_lookup["Average_Daily_Customers"] /
    agent_lookup["Average_Daily_Customers"].sum()
)

agent_weights = agent_lookup["Selection_Weight"].to_numpy()

agent_indices = np.arange(len(agent_lookup))



# Calendar


calendar_dates = pd.date_range(
    START_DATE,
    END_DATE,
    freq="D"
)


# Hours


hour_distribution = {

8:0.04,
9:0.06,
10:0.08,
11:0.09,
12:0.09,
13:0.08,
14:0.08,
15:0.10,
16:0.12,
17:0.13,
18:0.08,
19:0.04,
20:0.01

}

hours = list(hour_distribution.keys())

hour_probs = list(hour_distribution.values())

print("✓ Transaction Engine Ready")

✓ Transaction Engine Ready


In [49]:
required = [

"commission_rates",
"processing_cost",
"failure_rates",
"fraud_rates",
"payment_names",
"payment_probs",
"segment_names",
"segment_probs",
"transaction_type_df",
"agent_lookup",
"agent_weights",
"calendar_dates",
"hours",
"hour_probs",
"day_multiplier",
"salary_multiplier",
"generate_amount",
"calculate_commission",
"calculate_processing_cost",
"transaction_status",
"fraud_status"

]

missing = []

for item in required:

    if item not in globals():

        missing.append(item)

print("Missing:", missing)

Missing: []


In [ ]:

# Main Transaction Generator


transaction_chunks = []

transaction_id = 1

for chunk in tqdm(range(NUM_CHUNKS), desc="Generating Transactions"):

    rows = []

    for _ in range(CHUNK_SIZE):

       
        # Select Agent (Weighted)
    

        idx = np.random.choice(
            agent_indices,
            p=agent_weights
        )

        agent = agent_lookup.iloc[idx]

      
        # Random Date
       

        transaction_date = pd.Timestamp(
    np.random.choice(calendar_dates)
)

        weekday_factor = day_multiplier(transaction_date)

        salary_factor = salary_multiplier(transaction_date)

     
        # Hour
     

        hour = np.random.choice(
            hours,
            p=hour_probs
        )

        minute = np.random.randint(0,60)

        second = np.random.randint(0,60)

        timestamp = pd.Timestamp(
            year=transaction_date.year,
            month=transaction_date.month,
            day=transaction_date.day,
            hour=int(hour),
            minute=int(minute),
            second=int(second)
        )

      
        # Transaction Type
      

        tx_type = np.random.choice(
            transaction_type_df["Transaction_Type"],
            p=transaction_type_df["Probability"]
        )

       
        # Payment Method
       

        payment_method = np.random.choice(
            payment_names,
            p=payment_probs
        )

      
        # Customer Segment
        

        customer_segment = np.random.choice(
            segment_names,
            p=segment_probs
        )

     
        # Amount
       

        amount = generate_amount(tx_type)

        amount = round(
            amount * weekday_factor * salary_factor,
            2
        )

        
        # Revenue
       

        commission = calculate_commission(
            agent["Provider"],
            tx_type,
            amount
        )

        cost = calculate_processing_cost(
            tx_type
        )

        gross = commission

        net = gross - cost

       
        # Status
        

        status = transaction_status(
            agent["Provider"]
        )

        fraud = fraud_status(
            tx_type
        )

        rows.append({

            "Transaction_ID":f"TX{transaction_id:08d}",

            "Timestamp":timestamp,

            "Transaction_Date":timestamp.date(),

            "Hour":timestamp.hour,

            "Weekday":timestamp.day_name(),

            "Weekend":timestamp.day_name() in ["Saturday","Sunday"],

            "Agent_ID":agent["Agent_ID"],

            "Provider":agent["Provider"],

            "State":agent["State"],

            "Geo_Zone":agent["Geo_Zone"],

            "LGA":agent["LGA"],

            "Business_Category":agent["Business_Category"],

            "Agent_Tier":agent["Agent_Tier"],

            "Customer_Segment":customer_segment,

            "Payment_Method":payment_method,

            "Transaction_Type":tx_type,

            "Transaction_Amount":amount,

            "Commission":commission,

            "Processing_Cost":cost,

            "Gross_Revenue":gross,

            "Net_Revenue":net,

            "Transaction_Status":status,

            "Fraud_Status":fraud

        })

        transaction_id += 1

    chunk_df = pd.DataFrame(rows)

    transaction_chunks.append(chunk_df)

print("Generation Complete")

Generating Transactions:   0%|          | 0/10 [00:00<?, ?it/s]

In [94]:

# Merge All Chunks


fact_transactions = pd.concat(
    transaction_chunks,
    ignore_index=True
)

print(fact_transactions.shape)

fact_transactions.head()

(1000000, 23)


,Transaction_ID,Timestamp,Transaction_Date,Hour,Weekday,Weekend,Agent_ID,Provider,State,Geo_Zone,...,Customer_Segment,Payment_Method,Transaction_Type,Transaction_Amount,Commission,Processing_Cost,Gross_Revenue,Net_Revenue,Transaction_Status,Fraud_Status
0,TX00000001,2024-06-18 08:30:07,2024-06-18,8,Tuesday,False,AG004264,Moniepoint,Kano,North West,...,Retail,Card,Cash Deposit,24154.16,132.85,25,132.85,107.85,Successful,Clean
1,TX00000002,2024-02-27 18:11:28,2024-02-27,18,Tuesday,False,AG010115,OPay,Gombe,North East,...,Retail,Card,Cash Deposit,117374.26,586.87,25,586.87,561.87,Successful,Clean
2,TX00000003,2024-07-23 14:10:41,2024-07-23,14,Tuesday,False,AG002080,PalmPay,Lagos,South West,...,Retail,Card,Cash Deposit,90469.58,470.44,25,470.44,445.44,Successful,Clean
3,TX00000004,2024-10-12 16:18:16,2024-10-12,16,Saturday,True,AG008213,Moniepoint,Bauchi,North East,...,SME,Card,Bank Transfer,53532.61,214.13,18,214.13,196.13,Successful,Clean
4,TX00000005,2024-01-04 17:01:51,2024-01-04,17,Thursday,False,AG003155,Moniepoint,FCT,North Central,...,VIP,Card,Cash Deposit,79795.75,438.88,25,438.88,413.88,Successful,Clean


In [95]:

#Validation

print()

print("="*60)

print("Transaction Summary")

print("="*60)

print()

print("Rows :", len(fact_transactions))

print("Columns :", len(fact_transactions.columns))

print()

print("Missing Values")

print(fact_transactions.isna().sum())

print()

print("Duplicate IDs")

print(fact_transactions["Transaction_ID"].duplicated().sum())

print()

print("Status Distribution")

print(fact_transactions["Transaction_Status"].value_counts())

print()

print("Fraud Distribution")

print(fact_transactions["Fraud_Status"].value_counts())


Transaction Summary

Rows : 1000000
Columns : 23

Missing Values
Transaction_ID        0
Timestamp             0
Transaction_Date      0
Hour                  0
Weekday               0
Weekend               0
Agent_ID              0
Provider              0
State                 0
Geo_Zone              0
LGA                   0
Business_Category     0
Agent_Tier            0
Customer_Segment      0
Payment_Method        0
Transaction_Type      0
Transaction_Amount    0
Commission            0
Processing_Cost       0
Gross_Revenue         0
Net_Revenue           0
Transaction_Status    0
Fraud_Status          0
dtype: int64

Duplicate IDs
0

Status Distribution
Transaction_Status
Successful    984067
Failed         15933
Name: count, dtype: int64

Fraud Distribution
Fraud_Status
Clean    996804
Fraud      3196
Name: count, dtype: int64


In [102]:
print(type(provider_df))
print(type(location_df))
print(type(lga_df))
print(type(agents_df))
print(type(transaction_type_df))

<class 'pandas.core.frame.DataFrame'>
<class 'pandas.core.frame.DataFrame'>
<class 'pandas.core.frame.DataFrame'>
<class 'pandas.core.frame.DataFrame'>
<class 'pandas.core.frame.DataFrame'>


In [104]:

# SAVE DIMENSION TABLES


import os

# Create output folder if it doesn't exist
OUTPUT_FOLDER = "output"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# Save Provider Dimension
provider_df.to_csv(
    os.path.join(OUTPUT_FOLDER, "dim_provider.csv"),
    index=False
)

# Save Location Dimension
location_df.to_csv(
    os.path.join(OUTPUT_FOLDER, "dim_location.csv"),
    index=False
)

# Save LGA Dimension
lga_df.to_csv(
    os.path.join(OUTPUT_FOLDER, "dim_lga.csv"),
    index=False
)

# Save Agent Dimension
agents_df.to_csv(
    os.path.join(OUTPUT_FOLDER, "dim_agent.csv"),
    index=False
)

# Save Transaction Type Dimension
transaction_type_df.to_csv(
    os.path.join(OUTPUT_FOLDER, "dim_transaction_type.csv"),
    index=False
)

print("=" * 50)
print("Dimension tables exported successfully.")


for file in sorted(os.listdir(OUTPUT_FOLDER)):
    print(file)

Dimension tables exported successfully.
dim_agent.csv
dim_lga.csv
dim_location.csv
dim_provider.csv
dim_transaction_type.csv


In [106]:

# SAVE FACT TABLE


fact_transactions.to_csv(

    os.path.join(
        OUTPUT_FOLDER,
        "fact_transactions.csv"
    ),

    index=False

)

print("Fact table exported successfully.")

Fact table exported successfully.


In [108]:

# DATE DIMENSION


date_df = pd.DataFrame({

    "Date": pd.date_range(
        START_DATE,
        END_DATE,
        freq="D"
    )

})

date_df["Year"] = date_df["Date"].dt.year

date_df["Quarter"] = "Q" + date_df["Date"].dt.quarter.astype(str)

date_df["Month_Number"] = date_df["Date"].dt.month

date_df["Month_Name"] = date_df["Date"].dt.strftime("%B")

date_df["Month_Short"] = date_df["Date"].dt.strftime("%b")

date_df["Week_Number"] = date_df["Date"].dt.isocalendar().week.astype(int)

date_df["Day"] = date_df["Date"].dt.day

date_df["Day_Name"] = date_df["Date"].dt.day_name()

date_df["Weekend"] = date_df["Day_Name"].isin(
    ["Saturday","Sunday"]
)

date_df["YearMonth"] = (
    date_df["Date"].dt.strftime("%Y-%m")
)

date_df["MonthYear"] = (
    date_df["Date"].dt.strftime("%b-%Y")
)

date_df.to_csv(

    os.path.join(
        OUTPUT_FOLDER,
        "dim_date.csv"
    ),

    index=False

)

print(date_df.head())

        Date  Year Quarter  Month_Number Month_Name Month_Short  Week_Number  \
0 2024-01-01  2024      Q1             1    January         Jan            1   
1 2024-01-02  2024      Q1             1    January         Jan            1   
2 2024-01-03  2024      Q1             1    January         Jan            1   
3 2024-01-04  2024      Q1             1    January         Jan            1   
4 2024-01-05  2024      Q1             1    January         Jan            1   

   Day   Day_Name  Weekend YearMonth MonthYear  
0    1     Monday    False   2024-01  Jan-2024  
1    2    Tuesday    False   2024-01  Jan-2024  
2    3  Wednesday    False   2024-01  Jan-2024  
3    4   Thursday    False   2024-01  Jan-2024  
4    5     Friday    False   2024-01  Jan-2024  


In [110]:

# DAILY KPI


daily_kpi = (

    fact_transactions

    .groupby("Transaction_Date")

    .agg(

        Transactions=("Transaction_ID","count"),

        Revenue=("Gross_Revenue","sum"),

        Net_Revenue=("Net_Revenue","sum"),

        Volume=("Transaction_Amount","sum"),

        Failed=("Transaction_Status",

                lambda x:(x=="Failed").sum()),

        Fraud=("Fraud_Status",

               lambda x:(x=="Fraud").sum())

    )

    .reset_index()

)

daily_kpi.to_csv(

    os.path.join(
        OUTPUT_FOLDER,
        "kpi_daily.csv"
    ),

    index=False

)

daily_kpi.head()

,Transaction_Date,Transactions,Revenue,Net_Revenue,Volume,Failed,Fraud
0,2024-01-01,2725,878942.54,802944.54,1.379581e+08,34,9
1,2024-01-02,2864,782793.62,702433.62,1.223604e+08,46,16
2,2024-01-03,2861,797454.59,716713.59,1.242511e+08,48,13
3,2024-01-04,2662,721786.48,646937.48,1.125185e+08,35,7
4,2024-01-05,2745,844128.78,768127.78,1.308443e+08,40,12


In [112]:

# PROVIDER KPI


provider_kpi = (

    fact_transactions

    .groupby("Provider")

    .agg(

        Transactions=("Transaction_ID","count"),

        Revenue=("Gross_Revenue","sum"),

        Net_Revenue=("Net_Revenue","sum"),

        Volume=("Transaction_Amount","sum")

    )

    .reset_index()

)

provider_kpi.to_csv(

    os.path.join(
        OUTPUT_FOLDER,
        "kpi_provider.csv"
    ),

    index=False

)

provider_kpi

,Provider,Transactions,Revenue,Net_Revenue,Volume
0,FirstMonie,106294,31257352.00,28275720.00,4.675825e+09
1,Moniepoint,338519,96441583.52,86951325.52,1.491996e+10
2,OPay,295983,79501499.76,71207474.76,1.304522e+10
3,Paga,81437,24888753.09,22606967.09,3.596065e+09
4,PalmPay,177767,48913218.50,43921848.50,7.794600e+09


In [114]:

# STATE KPI


state_kpi = (

    fact_transactions

    .groupby("State")

    .agg(

        Transactions=("Transaction_ID","count"),

        Revenue=("Gross_Revenue","sum"),

        Net_Revenue=("Net_Revenue","sum"),

        Volume=("Transaction_Amount","sum")

    )

    .reset_index()

)

state_kpi.to_csv(

    os.path.join(
        OUTPUT_FOLDER,
        "kpi_state.csv"
    ),

    index=False

)

state_kpi.head()

,State,Transactions,Revenue,Net_Revenue,Volume
0,Abia,20104,5607258.58,5045738.58,8.860222e+08
1,Adamawa,17147,4798426.95,4319388.95,7.465545e+08
2,Akwa Ibom,29250,8235163.02,7414779.02,1.296427e+09
3,Anambra,31542,8874631.78,7990467.78,1.392559e+09
4,Bauchi,20752,5839419.78,5254701.78,9.119883e+08


In [116]:

# TRANSACTION TYPE KPI


transaction_kpi = (

    fact_transactions

    .groupby("Transaction_Type")

    .agg(

        Transactions=("Transaction_ID","count"),

        Revenue=("Gross_Revenue","sum"),

        Net_Revenue=("Net_Revenue","sum"),

        Volume=("Transaction_Amount","sum")

    )

    .reset_index()

)

transaction_kpi.to_csv(

    os.path.join(
        OUTPUT_FOLDER,
        "kpi_transaction_type.csv"
    ),

    index=False

)

transaction_kpi

,Transaction_Type,Transactions,Revenue,Net_Revenue,Volume
0,Airtime,70082,5.239372e+06,4.538552e+06,2.660325e+08
1,Bank Transfer,159127,5.491798e+07,5.205369e+07,1.374289e+10
2,Bill Payment,40045,1.278095e+07,1.218028e+07,7.200232e+08
3,Cash Deposit,170282,4.929757e+07,4.504052e+07,9.178327e+09
4,Cash Withdrawal,560464,1.587665e+08,1.391503e+08,2.012440e+10


In [118]:

# CUSTOMER SEGMENT KPI


segment_kpi = (

    fact_transactions

    .groupby("Customer_Segment")

    .agg(

        Transactions=("Transaction_ID","count"),

        Revenue=("Gross_Revenue","sum"),

        Net_Revenue=("Net_Revenue","sum"),

        Volume=("Transaction_Amount","sum")

    )

    .reset_index()

)

segment_kpi.to_csv(

    os.path.join(
        OUTPUT_FOLDER,
        "kpi_customer_segment.csv"
    ),

    index=False

)

segment_kpi

,Customer_Segment,Transactions,Revenue,Net_Revenue,Volume
0,Corporate,59945,1.677349e+07,1.509082e+07,2.628794e+09
1,Retail,719627,2.021874e+08,1.820179e+08,3.168520e+10
2,SME,190382,5.359168e+07,4.824932e+07,8.396814e+09
3,VIP,30046,8.449834e+06,7.605255e+06,1.320860e+09


In [120]:

# LGA KPI


lga_kpi = (

    fact_transactions

    .groupby(["State", "LGA"])

    .agg(

        Transactions=("Transaction_ID","count"),

        Revenue=("Gross_Revenue","sum"),

        Net_Revenue=("Net_Revenue","sum"),

        Volume=("Transaction_Amount","sum")

    )

    .reset_index()

)

lga_kpi.to_csv(

    os.path.join(
        OUTPUT_FOLDER,
        "kpi_lga.csv"
    ),

    index=False

)

lga_kpi.head()

,State,LGA,Transactions,Revenue,Net_Revenue,Volume
0,Abia,Abia Central,3807,1061943.82,955852.82,1.699557e+08
1,Abia,Abia East,5248,1469978.47,1322760.47,2.321704e+08
2,Abia,Abia North,5082,1419868.18,1278214.18,2.228168e+08
3,Abia,Abia South,2881,786015.24,705628.24,1.240409e+08
4,Abia,Abia West,3086,869452.87,783282.87,1.370383e+08


In [122]:

# EXPORT SUMMARY


print("=" * 60)
print("EXPORT COMPLETE")
print("=" * 60)

files = sorted(os.listdir(OUTPUT_FOLDER))

for file in files:
    print(file)

print("\nTotal Files:", len(files))

EXPORT COMPLETE
dim_agent.csv
dim_date.csv
dim_lga.csv
dim_location.csv
dim_provider.csv
dim_transaction_type.csv
fact_transactions.csv
kpi_customer_segment.csv
kpi_daily.csv
kpi_lga.csv
kpi_provider.csv
kpi_state.csv
kpi_transaction_type.csv

Total Files: 13
